# Milestone - Unit 8: system identification

Unit 8 asked what the car really is, instead of what Unit 3 guessed, and measured it: the longitudinal map and the steering gain from logs with least squares, and the grip of each section from limit probes.
`agents/unit-08.py` is Unit 7's file with what was measured:

- `MAX_STEER = math.radians(66.0)`: the steady log's fit measured 65.7° against the 70° Unit 7 assumed, and the file rounds it to a whole degree;
- `MU_G`, six measured μ̂·g values (m/s², one per grip section; the file ships `test-circuit`'s), and `MU_FRACTION = 0.56`: the corner cap is √(`MU_FRACTION` · `MU_G`[s] · `GRIP`[s] / |κ_i|), one grid step inside the training family's clean edge, 0.58;
- with `MU_G = None` the cap is Unit 7's √(A_LAT · GRIP[s] / |κ_i|).

The notebook asserts the unit's gates (plan `roar-008`) on the trimmed subset below; a routed racekit test (`tests/test_racekit_unit08_09_premise.py`) races every multi-seed figure in full.
A claim "not clean on seeds 0–4" is proven by one seed that is not clean, so the milestone races that seed.

- **U8.1** the committed logs: the longitudinal fit's residual is smallest at lag 1, and its coefficients, and the steady log's steering gain, 65.71°, are reproduced;
- **U8.2** the grip probe: the six `test-circuit` probes reproduce the shipped `MU_G` to 0.01 m/s², and every one of them crashed (so each is an estimate); `monza-like`'s section 0 probe has 0 collisions (so its number is a lower bound);
- **U8.3** the steering gain: on `monza-like` the measured-grip planner at f = 0.54 is not clean with the assumed 70° (seed 1) and clean at 66° up to f = 0.62 (seed 0, 497.90 s); the Unit 7 reference at 66° is still not clean on `test-circuit-seed2026` (seed 0);
- **U8.4** the transfer: at 66° with each track's own `MU_G`, f = 0.58 is clean on each of the five training tracks (seed 0), 0.60 is not clean on `test-circuit-seed4` (seed 0); the frozen 0.56 is clean on `test-circuit-seed2026` on seeds 0–4, median 303.30 s, and 303.30 + 3 × the larger spread is below Checkpoint A's 310.80 s;
- **U8.5** the reference: `agents/unit-08.py` on `test-circuit`, seeds 0–4, all clean, median 301.95 s; `check_submission` passes; with `MU_G = None` it races exactly like `agents/unit-07.py` with a 66° `MAX_STEER` (seed 0, the same telemetry to the tick).

Every race uses racekit's carla-like car, `carla_like(noise=True)`, seeded by `run_race`, capped at 450 s on a `test-circuit` track and 900 s on `monza-like` (a probe: one lap, 300 s, seed 0).
Tolerances (the plan's Gates header): race times and medians within one tick (±0.05 s); an unclean race is asserted only "not clean", never a collision count.
A race is **clean** when it finishes with 0 collisions and its max |cross-track error| on driving rows is at most 2 m.
Every figure is a model number on racekit's carla-like car, not CARLA's; no figure here is compared with the car's hidden parameters.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
A variant is a reference file with some constants changed (`write_variant`), written to a temporary directory; a track's measured grip is written in exactly as Checkpoint A's settings were.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.sysid_data import load_log
from racekit.tracks import grip_sections, load_track

AGENTS = Path("..") / "agents"
UNIT_07, UNIT_08 = AGENTS / "unit-07.py", AGENTS / "unit-08.py"
SEEDS = (0, 1, 2, 3, 4)
TICK = 0.05 + 1e-9
CLEAN_OFFSET = 2.0  # m
DT = 0.05  # s
CHECKPOINT_A = 310.80  # s: Checkpoint A's pinned median on test-circuit-seed2026 (plan roar-007)
CHECKPOINT_A_SPREAD = 0.20  # s: its pinned spread
SHIPPED_MU_G = (7.89, 9.72, 8.65, 7.87, 8.30, 8.14)  # agents/unit-08.py's MU_G (test-circuit)
MEASURED_MU_G = {  # the probes' mu * g per section (m/s^2, seed 0, rounded to 0.01; the lesson's table)
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed2": (9.06, 8.37, 8.49, 8.86, 9.78, 9.92),
    "test-circuit-seed3": (9.55, 8.89, 8.15, 9.94, 8.50, 9.09),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
    "monza-like": (5.38, 8.96, 9.12, 9.14, 9.70, 7.97),
}
workdir = Path(mkdtemp(prefix="roar-milestone-08-"))


def variant(source, name, **constants):
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


def cap_for(track):
    return 900.0 if track == "monza-like" else 450.0


def race(source, track="test-circuit", seed=0):
    return run_race(source, track, car=carla_like(noise=True), seed=seed, max_seconds=cap_for(track))


def seeds_race(source, track="test-circuit"):
    return repeat_races(source, track, n=len(SEEDS), seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                        max_seconds=cap_for(track))


def measured(track, fraction, steer_deg=66.0):
    # agents/unit-08.py with a track's measured grip, a fraction and a steering gain (66 deg is the shipped one).
    constants = {"MU_G": MEASURED_MU_G[track], "MU_FRACTION": fraction}
    if steer_deg != 66.0:
        constants["MAX_STEER"] = math.radians(steer_deg)
    return variant(UNIT_08, f"u08-{track}-{fraction}-{steer_deg}", **constants)

## 2. U8.1: the committed logs

The lesson's two fits, on `racekit.sysid_data`'s logs: the longitudinal map dv/dt = a_T·throttle − a_B·brake − c·v|v| by least squares for lags 0–3 (pairs of rows one tick apart, driven by the command logged `lag` ticks earlier), and the steering gain from the steady rows of the oval log (the lagged steer still to σ < 0.002 over 11 ticks, |steer| > 0.005, v > 5 m/s), wheel angle atan(r·L / v) = MAX_STEER · (−steer).

In [ ]:
race_log, race_meta = load_log("monza-like-seed0.csv")
steady_log, steady_meta = load_log("oval-steady-seed0.csv")
assert race_meta["kind"] == "race" and steady_meta["kind"] == "steady"


def longitudinal_fit(log, lag):
    speed = np.hypot(log["vx"], log["vy"])
    k = np.arange(lag, len(log) - 1)
    keep = (np.abs(log["t"][k + 1] - log["t"][k] - DT) < 1e-6) & (np.abs(log["t"][k] - log["t"][k - lag] - lag * DT) < 1e-6)
    k = k[keep]
    mean = (speed[k] + speed[k + 1]) / 2.0
    a = np.column_stack([log["throttle"][k - lag], -log["brake"][k - lag], -mean * np.abs(mean)])
    b = (speed[k + 1] - speed[k]) / DT
    c = np.linalg.lstsq(a, b, rcond=None)[0]
    return c, float(np.sqrt(np.mean((a @ c - b) ** 2)))


fits = {lag: longitudinal_fit(race_log, lag) for lag in range(4)}
for lag, (c, residual) in fits.items():
    print(f"lag {lag}: a_T {c[0]:.3f}, a_B {c[1]:.3f}, c {c[2]:.3e}, residual RMS {residual:.3f} m/s^2")
assert min(fits, key=lambda lag: fits[lag][1]) == 1
a_throttle, a_brake, c_drag = fits[1][0]
assert abs(a_throttle - 5.174) < 0.001 and abs(a_brake - 8.539) < 0.001 and abs(c_drag - 3.081e-4) < 1e-7

yaw = steady_log["yaw"]
rate = ((yaw[1:] - yaw[:-1] + math.pi) % (2.0 * math.pi) - math.pi) / DT
speed = np.hypot(steady_log["vx"], steady_log["vy"])
mean_speed = (speed[:-1] + speed[1:]) / 2.0
applied = np.concatenate([[0.0], steady_log["steer"][:-2]])
stillness = np.array([np.std(applied[max(0, k - 10):k + 1]) for k in range(len(applied))])
steady = (mean_speed > 5.0) & (np.abs(applied) > 0.005) & (stillness < 0.002)
wheel_angle = np.array([math.atan2(r_k * 2.875, v_k) for r_k, v_k in zip(rate[steady], mean_speed[steady])])
gain = math.degrees(np.linalg.lstsq(np.column_stack([-applied[steady]]), wheel_angle, rcond=None)[0][0])
print(f"steering gain: {gain:.2f} deg from {int(np.sum(steady))} steady rows (Unit 7 assumed 70 deg)")
assert abs(gain - 65.71) < 0.005 and round(gain) == 66

## 3. U8.2: the grip probe

The student's own Unit 7 agent as the driver: `A_LAT = 3.5`, and `GRIP[s] = 3.5` in one section s, one lap, 300 s, seed 0.
|a| is the second difference of the telemetry's (x, y) at 0.05 s, over rows whose three consecutive rows are driving rows (`held == 0`) with no step over 5 m and v > 5 m/s, in the section of waypoint `(progress + 1) mod n`; the estimate is the 99th percentile, and a section with 20 qualifying rows or fewer gives none.

In [ ]:
def probe(track, section):
    grip = [1.0] * 6
    grip[section] = 3.5
    source = variant(UNIT_07, f"probe-{track}-{section}", A_LAT=3.5, GRIP=tuple(grip))
    return run_race(source, track, car=carla_like(noise=True), seed=0, max_seconds=300.0, laps=1)


def pushed_section(result, track, section):
    # (qualifying rows, the 99th percentile of |a| or None) in the pushed section of a probe race
    telemetry = result.telemetry
    x, y, held = telemetry["x"], telemetry["y"], telemetry["held"]
    accel = np.hypot((x[2:] - 2.0 * x[1:-1] + x[:-2]) / DT ** 2, (y[2:] - 2.0 * y[1:-1] + y[:-2]) / DT ** 2)
    step = np.hypot(np.diff(x), np.diff(y))
    ok = (held[:-2] == 0) & (held[1:-1] == 0) & (held[2:] == 0)
    ok = ok & (step[:-1] < 5.0) & (step[1:] < 5.0) & (telemetry["v"][1:-1] > 5.0)
    sections = np.array(grip_sections(load_track(track)))
    in_section = sections[(telemetry["progress"][1:-1] + 1) % len(sections)] == section
    values = accel[ok & in_section]
    return len(values), (float(np.percentile(values, 99.0)) if len(values) > 20 else None)


estimates = []
for s in range(6):
    result = probe("test-circuit", s)
    rows, value = pushed_section(result, "test-circuit", s)
    estimates.append(round(value, 2))
    print(f"test-circuit section {s}: {result.collisions} collisions, {rows} rows, mu * g = {value:.2f} m/s^2")
    assert result.collisions >= 1  # the probe slid: an estimate
assert tuple(estimates) == SHIPPED_MU_G

with shim_installed():
    Unit08, _ = load_submission(UNIT_08)
agent = Unit08(load_track("test-circuit").waypoints, None)
assert all(abs(agent.lateral_limit(s) - 0.56 * SHIPPED_MU_G[s]) < 1e-12 for s in range(6))  # the shipped MU_G
print("the probes reproduce agents/unit-08.py's MU_G:", tuple(estimates))

`monza-like`'s section 0 is the start straight: its probe never slides, so its number is a lower bound, and the agent may plan with it as measured (it can only be too cautious).

In [ ]:
start = probe("monza-like", 0)
rows, value = pushed_section(start, "monza-like", 0)
print(f"monza-like section 0: {start.collisions} collisions, {rows} rows, 99th percentile {value:.2f} m/s^2 "
      "(a lower bound)")
assert start.collisions == 0 and rows > 20 and round(value, 2) == MEASURED_MU_G["monza-like"][0]

## 4. U8.3: the steering gain

The measured-grip planner (`agents/unit-08.py` with `monza-like`'s `MU_G`) at f = 0.54 with the assumed 70°, on seed 1, and at 66° with f = 0.62, the top of its clean range, on seed 0.
Then the Unit 7 reference with only the steering gain fixed, on `test-circuit-seed2026`, seed 0.

In [ ]:
at_70 = race(measured("monza-like", 0.54, steer_deg=70.0), track="monza-like", seed=1)
print(f"monza-like, f 0.54, 70 deg, seed 1: clean {clean(at_70)}")
assert not clean(at_70)

top = race(measured("monza-like", 0.62), track="monza-like", seed=0)
print(f"monza-like, f 0.62, 66 deg, seed 0: clean {clean(top)}, {top.total_time:.2f} s")
assert clean(top) and abs(top.total_time - 497.90) <= TICK

unit_7_at_66 = race(variant(UNIT_07, "unit-07-66deg", MAX_STEER=math.radians(66.0)), track="test-circuit-seed2026")
print(f"Unit 7 at 66 deg, test-circuit-seed2026, seed 0: clean {clean(unit_7_at_66)}")
assert not clean(unit_7_at_66)

## 5. U8.4: the family edge and the transfer

At 66° with each track's own measured grip: f = 0.58 on each training track (seed 0; the routed test races seeds 0–4), and 0.60 on `test-circuit-seed4` (seed 0).
`test-circuit` races its shipped `MU_G`, so its variant changes only `MU_FRACTION`.

In [ ]:
EDGE_SEED_0 = {  # f = 0.58, race seed 0 (probe record, plan roar-008 p8)
    "test-circuit": 299.40, "test-circuit-seed1": 295.15, "test-circuit-seed2": 294.25,
    "test-circuit-seed3": 294.75, "test-circuit-seed4": 295.75,
}
for track, pin in EDGE_SEED_0.items():
    if track == "test-circuit":
        source = variant(UNIT_08, "u08-test-circuit-0.58", MU_FRACTION=0.58)
    else:
        source = measured(track, 0.58)
    result = race(source, track=track)
    print(f"{track:22s} f 0.58, seed 0: clean {clean(result)}, {result.total_time:.2f} s")
    assert clean(result) and abs(result.total_time - pin) <= TICK

past_edge = race(measured("test-circuit-seed4", 0.60), track="test-circuit-seed4")
print(f"test-circuit-seed4     f 0.60, seed 0: clean {clean(past_edge)}")
assert not clean(past_edge)

The frozen f = 0.56 on the unseen `test-circuit-seed2026`, with its own probed grip, seeds 0–4, against Checkpoint A's tuned global setting on the same track.

In [ ]:
transfer = seeds_race(measured("test-circuit-seed2026", 0.56), track="test-circuit-seed2026")
print("test-circuit-seed2026, f 0.56:", [f"{time:.2f}" for time in transfer.times],
      f"median {transfer.median:.2f} s (spread {transfer.spread:.2f})")
assert all(clean(result) for result in transfer.results)
assert abs(transfer.median - 303.30) <= TICK
noise = max(transfer.spread, CHECKPOINT_A_SPREAD)
print(f"{transfer.median:.2f} + 3 x {noise:.2f} = {transfer.median + 3 * noise:.2f} s < Checkpoint A's {CHECKPOINT_A:.2f} s")
assert transfer.median + 3 * noise < CHECKPOINT_A

## 6. U8.5: the reference, and the submission checks

Five seeds on `test-circuit`, `check_submission`, and the equivalence: with `MU_G = None` the file is Unit 7's with a 66° gain, raced on seed 0 with identical telemetry.

In [ ]:
reference = seeds_race(UNIT_08)
print("agents/unit-08.py, seeds 0-4:", [f"{time:.2f}" for time in reference.times],
      f"median {reference.median:.2f} s (spread {reference.spread:.2f})")
assert all(clean(result) for result in reference.results)
assert abs(reference.median - 301.95) <= TICK

report = check_submission(UNIT_08)
print(report)
assert report.ok, report.findings

no_mu = race(variant(UNIT_08, "u08-mu-g-none", MU_G=None))
unit_7_66_home = race(variant(UNIT_07, "u07-66deg-home", MAX_STEER=math.radians(66.0)))
print(f"MU_G = None: {no_mu.total_time:.2f} s; Unit 7 at 66 deg: {unit_7_66_home.total_time:.2f} s")
assert no_mu.total_time == unit_7_66_home.total_time and no_mu.collisions == unit_7_66_home.collisions
assert no_mu.telemetry.tobytes() == unit_7_66_home.telemetry.tobytes()

## Unit 8, pinned

| on the carla-like car | `test-circuit` | `test-circuit-seed2026` (unseen) | `monza-like` |
|---|---|---|---|
| Unit 7 (70°, A_LAT 4.6) | 304.15 s | not clean | 515.10 s |
| Checkpoint A (tuned global setting) | | 310.80 s | |
| **Unit 8** (66°, f = 0.56 of each track's measured grip) | **301.95 s** | **303.30 s** | 506.50 s (clean up to f = 0.62, 497.90 s) |

The submission ships `test-circuit`'s `MU_G`: the grip is per track, and the probe measures it on any new track in six one-lap races.
The Unit 8 lab does the same identification on CARLA, and races only a fraction it has raced clean twice.